In [ ]:
import sympy
import numpy as np

In [ ]:
A = sympy.symbols([f"a_{{{i}{j}}}" for i in range(1, 4) for j in "xyz"])
A = np.array(A).reshape(3, 3)
display(sympy.Matrix(A))

In [ ]:
def frobenius_norm(A):
    sum = 0
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            sum += A[i, j]**2
    return sympy.sqrt(sum)

standard_version = frobenius_norm(A @ A.T - np.eye(3))**2

In [ ]:
paper_version = sum((A[i].dot(A[i]) - 1)**2 for i in range(3)) \
 + sum((A[i].dot(A[j]))**2 for i in range(3) for j in range(3) if i != j)

In [ ]:
(standard_version - paper_version).simplify()

In [ ]:
for i in range(9):
    for j in range(9):
        display(standard_version.diff(A[i//3, i%3]).diff(A[j//3, j%3]))

In [ ]:
x = np.array(sympy.symbols([f"{d}{i}" for i in range(1, 5) for d in "xyz"])).reshape(-1, 3)
display(sympy.Matrix(x))

In [ ]:
p = np.array(sympy.symbols("p_x p_y p_z"))
dof = np.hstack((p, A.flatten(order="C")))

In [ ]:
J = np.zeros((x.size, 12), dtype=object)

for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        J[x.shape[0]*j + i, j] = 1

# J[:, 3:] = np.kron(np.eye(3, dtype=int), x)
for i in range(x.shape[0]):
    for j in range(x.shape[1]):
        for k in range(A.shape[0]):
            J[i + k * x.shape[0], j + k * x.shape[1] + p.size] = x[i, j]

display(sympy.Matrix(J))

linear_version = J @ dof
affine_version = (x @ A.T + p).flatten(order="F")

display(sympy.Matrix(linear_version))
display(sympy.Matrix(affine_version))
display(sympy.Matrix(linear_version - affine_version))

display(sympy.Matrix(linear_version).diff(dof))

In [ ]:
sympy.Function("f")(*(J @ dof)).diff(dof[0])

In [ ]:
display(sympy.Symbol(r"\rho") * sympy.Matrix(J.T @ J) / x.shape[0])